# ══════════════════════════════════════════════════════════════
#  WEEK 12  |  BONUS  |  TALK TO ANY DATABASE USING AI
# ══════════════════════════════════════════════════════════════
#
#  LEARNING OBJECTIVES
#  ───────────────────
#  1. Build a FastAPI service that accepts natural-language questions
#  2. Wire LangChain tool calling to a local SQLite database via Groq
#  3. Use Python decorators (@tool, @app.get) as production patterns
#  4. Apply context managers (with sqlite3.connect) for safe DB access
#  5. Add type hints to every function signature
#  6. Build a Streamlit frontend that queries the FastAPI backend
#
#  ARCHITECTURE
#  ────────────
#  Streamlit (app.py)
#      POST /query {"question": "..."}
#          FastAPI (backend.py)
#              LangChain AgentExecutor (Groq llama-3.1-70b-versatile)
#                  Tool 1: get_schema()  → reads SQLite schema
#                  Tool 2: run_sql(sql)  → executes SELECT only
#              {"sql": "SELECT ...", "results": [...]}
#          Streamlit: st.dataframe(results) + st.code(sql)
#
#  SETUP (run once before starting the app)
#  ─────────────────────────────────────────
#  pip install fastapi uvicorn langchain langchain-groq langchain-core
#              streamlit python-dotenv pydantic requests
#
#  1. Copy nl_sql_app/.env.example to nl_sql_app/.env, add your Groq key
#     Get a free key at: https://console.groq.com
#  2. python nl_sql_app/seed_data.py
#  3. uvicorn nl_sql_app.backend:app --reload --port 8000
#  4. streamlit run nl_sql_app/app.py   (separate terminal)
#
#  TIME:  ~60 minutes
#
#  ─────────────────────────────────────────────────────────────
#  ARCHITECTURE DECISION
#  ─────────────────────
#  Choosing between: OpenAI GPT-4o vs Groq llama-3.1-70b vs local Ollama.
#  Rule of thumb: use Groq for fast, free development — near-GPT-4 quality
#  at zero cost during prototyping. Move to GPT-4o for production accuracy.
#  Use Ollama when data must not leave the local machine (air-gapped systems).
#
# ══════════════════════════════════════════════════════════════

In [ ]:
import sqlite3
import json
import os
import time
import tempfile
from datetime import datetime

print("=" * 62)
print("  NL-SQL PROJECT — Notebook walkthrough")
print("=" * 62)
print()
print("This notebook explains the architecture of nl_sql_app/.")
print("Read each concept, then complete the TASK sections.")
print("The real app runs in backend.py + app.py.")

# ══════════════════════════════════════════════════════════════
#  CONCEPT 1 — PYTHON DECORATORS
# ══════════════════════════════════════════════════════════════
#
#  A decorator is a function that wraps another function to add
#  behavior before or after it runs — without modifying the
#  original function's code.
#
#  Syntax:
#    @decorator_name
#    def my_function():
#        ...
#
#  This is exactly equivalent to:
#    my_function = decorator_name(my_function)
#
#  The @ symbol is syntactic sugar. The decorator receives the
#  function as an argument and returns a (possibly modified) version.
#
#  In this project, decorators appear in two places:
#
#    1. @tool (from LangChain)
#       Registers a Python function as a callable tool that the LLM
#       can request to invoke. LangChain reads the function name,
#       type hints, and docstring to build the tool's schema.
#
#    2. @app.get("/health") and @app.post("/query") (from FastAPI)
#       Register a Python function as an HTTP route handler.
#       FastAPI reads the type hints and Pydantic return type to
#       generate automatic API documentation at /docs.
#
#  EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
import functools
import time as _time

# ── Decorator example: log_calls ─────────────────────────────
def log_calls(func):
    """Decorator: prints the function name and execution time."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        t0 = _time.time()
        result = func(*args, **kwargs)
        ms = round((_time.time() - t0) * 1000, 1)
        print(f"  [log_calls] {func.__name__}() ran in {ms} ms")
        return result
    return wrapper


@log_calls
def fetch_top_products(limit: int = 5) -> list:
    """Return the top N products."""
    products = ["Laptop Pro", "Monitor 27in", "Mechanical Keyboard",
                "USB Hub", "Webcam HD", "Mouse Wireless"]
    return products[:limit]


@log_calls
def count_users() -> int:
    """Return the user count."""
    return 10


products = fetch_top_products(3)
user_count = count_users()

print(f"\nTop 3 products : {products}")
print(f"User count     : {user_count}")
print()
print("@log_calls added timing output WITHOUT changing the function bodies.")
print("That is the decorator pattern.")

# ── How @tool and @app.post work in this project ─────────────
#
#  @tool (backend.py)
#  ------------------
#  @tool
#  def run_sql(sql: str) -> str:
#      """Run a SELECT query and return results as JSON.
#         Only SELECT queries are permitted."""
#      ...
#
#  LangChain reads the docstring to build the tool description.
#  The LLM receives: name, parameter schema, and description.
#  It decides which tool to call — your code runs the actual function.
#
#  @app.post (backend.py)
#  ----------------------
#  @app.post("/query", response_model=QueryResponse)
#  def query(req: QueryRequest) -> QueryResponse:
#      ...
#
#  FastAPI registers this as the handler for POST /query.
#  response_model=QueryResponse tells FastAPI to validate the
#  return value and document the response shape at /docs.

# ══════════════════════════════════════════════════════════════
#  CONCEPT 2 — CONTEXT MANAGERS
# ══════════════════════════════════════════════════════════════
#
#  A context manager defines what happens when you enter and exit
#  a 'with' block. Python guarantees the exit code runs even if
#  an exception is raised inside the block.
#
#  Syntax:
#    with some_resource as variable:
#        # use variable here
#    # resource is automatically released here, even on error
#
#  sqlite3.connect used as a context manager:
#    - On normal exit : commits the transaction
#    - On exception   : rolls back the transaction
#    - Always         : closes the connection
#
#  Without a context manager you must call conn.close() manually.
#  In production pipelines that run thousands of queries, a
#  leaked connection exhausts the pool and brings down the service.
#
#  EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
# ── Context manager demo ──────────────────────────────────────
with sqlite3.connect(":memory:") as demo_conn:
    cur = demo_conn.cursor()
    cur.execute("""
        CREATE TABLE products (
            id    INTEGER PRIMARY KEY,
            name  TEXT,
            price REAL
        )
    """)
    cur.executemany("INSERT INTO products VALUES (?,?,?)", [
        (1, "Laptop Pro",   1299.0),
        (2, "Monitor 27in",  399.0),
        (3, "USB Hub",        49.99),
    ])
    cur.execute("SELECT * FROM products WHERE price > 100")
    rows = cur.fetchall()

# Connection is closed here automatically.
print("Products above $100:")
for row in rows:
    print(f"  id={row[0]}  name={row[1]}  price=${row[2]:.2f}")

print()
print("The 'with' block closed the connection automatically.")
print("No conn.close() call was needed.")

# ══════════════════════════════════════════════════════════════
#  CONCEPT 3 — TYPE HINTS
# ══════════════════════════════════════════════════════════════
#
#  Type hints annotate variables and function signatures with the
#  expected type. Python does not enforce them at runtime, but:
#
#    - VS Code uses them to catch mistakes as you type
#    - FastAPI and Pydantic enforce them at runtime
#    - LangChain uses them to build tool parameter schemas
#
#  Syntax:
#    variable: type = value
#    def func(param: type) -> return_type:
#
#  Types used in this project:
#    str           plain string
#    int           integer
#    float         floating-point number
#    list[dict]    a list where every element is a dict
#    dict          a dictionary with unspecified key/value types
#
#  Pydantic enforces type hints at runtime:
#    class QueryRequest(BaseModel):
#        question: str         # FastAPI rejects non-string values
#    class QueryResponse(BaseModel):
#        sql:        str
#        results:    list[dict]
#        latency_ms: float
#
#  EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
# ── Type hints example ────────────────────────────────────────
def summarize_orders(orders: list, min_amount: float = 0.0) -> dict:
    """
    Summarize a list of order dicts.
    Returns count, total, and average for orders above min_amount.
    """
    filtered = [o for o in orders if o["amount"] >= min_amount]
    total: float = sum(o["amount"] for o in filtered)
    count: int = len(filtered)
    average: float = round(total / count, 2) if count else 0.0
    return {"count": count, "total": total, "average": average}


sample_orders: list = [
    {"id": 1, "product": "Laptop Pro",   "amount": 1299.0},
    {"id": 2, "product": "USB Hub",       "amount":   49.99},
    {"id": 3, "product": "Monitor 27in", "amount":  399.0},
    {"id": 4, "product": "Mouse",        "amount":   39.99},
]

all_orders: dict = summarize_orders(sample_orders)
big_orders: dict = summarize_orders(sample_orders, min_amount=100.0)

print(f"All orders    : {all_orders}")
print(f"Orders >= 100 : {big_orders}")
print()
print("Type hints make the intent clear:")
print("  orders: list  — the caller knows what to pass")
print("  -> dict        — the caller knows what to expect back")

# ══════════════════════════════════════════════════════════════
#  CONCEPT 4 — LANGCHAIN AGENTEXECUTOR WITH GROQ
# ══════════════════════════════════════════════════════════════
#
#  The AgentExecutor runs the ReAct loop:
#    LLM decides which tool to call
#    → your code runs it
#    → result goes back to the LLM
#    → repeat until the LLM has an answer
#
#  Pattern (used verbatim in backend.py):
#
#    from langchain_groq import ChatGroq
#    from langchain.tools import tool
#    from langchain.agents import create_tool_calling_agent, AgentExecutor
#    from langchain_core.prompts import ChatPromptTemplate
#
#    llm = ChatGroq(api_key=GROQ_API_KEY, model="llama-3.1-70b-versatile", temperature=0)
#
#    @tool
#    def get_schema() -> str:
#        """Return the database schema as JSON."""
#        ...
#
#    @tool
#    def run_sql(sql: str) -> str:
#        """Run a SELECT query. Only SELECT is allowed."""
#        ...
#
#    prompt = ChatPromptTemplate.from_messages([
#        ("system", "You are a SQL expert. Call get_schema first..."),
#        ("human", "{input}"),
#        ("placeholder", "{agent_scratchpad}"),
#    ])
#
#    executor = AgentExecutor(
#        agent=create_tool_calling_agent(llm, tools, prompt),
#        tools=[get_schema, run_sql],
#        verbose=False,
#        max_iterations=6,
#    )
#
#    result = executor.invoke({"input": "Which country has the most orders?"})
#    print(result["output"])   # plain English answer
#
#  GROQ vs OPENAI (one-line swap):
#    OpenAI: ChatOpenAI(model="gpt-4o-mini", api_key=...)
#    Groq:   ChatGroq(model="llama-3.1-70b-versatile", api_key=...)
#
#  EXAMPLE — simulated ReAct loop (no API key required) ─────────

In [ ]:
# ── Simulated AgentExecutor loop ──────────────────────────────
# Shows the THINK → ACT → OBSERVE pattern without an API key.

def fake_get_schema() -> str:
    schema = {
        "users":  [{"column": "id"}, {"column": "name"}, {"column": "city"}, {"column": "country"}],
        "orders": [{"column": "id"}, {"column": "user_id"}, {"column": "product"}, {"column": "amount"}],
    }
    return json.dumps(schema, indent=2)


def fake_run_sql(sql: str) -> str:
    if not sql.strip().upper().startswith("SELECT"):
        return json.dumps({"error": "Only SELECT queries are permitted."})
    return json.dumps([
        {"country": "Canada", "order_count": 8},
        {"country": "USA",    "order_count": 6},
    ])


def simulate_agent(question: str) -> None:
    print(f"Question: {question}")
    print("-" * 50)

    print("THINK: I need to know the database structure.")
    print("ACT:   get_schema()")
    schema_result = fake_get_schema()
    print(f"OBSERVE: {schema_result[:80]}...")

    sql = "SELECT u.country, COUNT(o.id) AS order_count FROM users u JOIN orders o ON u.id = o.user_id GROUP BY u.country ORDER BY order_count DESC"
    print(f"\nTHINK: I know the schema. I will write a JOIN query.")
    print(f"ACT:   run_sql(sql='{sql[:55]}...')")
    results = fake_run_sql(sql)
    print(f"OBSERVE: {results}")

    rows = json.loads(results)
    top = rows[0]
    print(f"\nFINAL ANSWER: {top['country']} has the most orders ({top['order_count']}).")


simulate_agent("Which country has the most orders?")

# ══════════════════════════════════════════════════════════════
#  TASK 1 — BUILD THE DATABASE SEED FUNCTION
# ══════════════════════════════════════════════════════════════
#
#  Write a function seed_demo_db(db_path: str) -> None that:
#
#    1. Uses 'with sqlite3.connect(db_path) as conn:'
#    2. Creates two tables:
#         employees (id INTEGER PRIMARY KEY, name TEXT, dept TEXT, salary REAL)
#         projects  (id INTEGER PRIMARY KEY, name TEXT, budget REAL, lead_id INTEGER)
#    3. Inserts the starting data below using parameterized queries
#    4. Prints how many rows were inserted for each table
#
#  Call seed_demo_db(':memory:') and then verify the data by querying
#  inside the same 'with' block.
#
#  Expected output:
#    Seeded: employees=4 rows, projects=4 rows
#    Employees with salary > 80000:
#      Alice — Engineering — $95000.0
#      Carol — Finance — $85000.0
#
# --- starting data ---

In [ ]:
employee_rows = [
    (1, "Alice", "Engineering", 95000),
    (2, "Bob",   "Sales",       72000),
    (3, "Carol", "Finance",     85000),
    (4, "Dave",  "Engineering", 78000),
]

project_rows = [
    (1, "Data Platform",    120000, 1),
    (2, "CRM Migration",     80000, 2),
    (3, "Analytics Hub",     95000, 3),
    (4, "API Modernization", 70000, 4),
]

# ══════════════════════════════════════════════════════════════
#  TASK 2 — DEFINE THE LANGCHAIN TOOLS
# ══════════════════════════════════════════════════════════════
#
#  Define two @tool functions that match the backend.py pattern.
#  Both functions must use 'with sqlite3.connect(task2_db) as conn:'
#
#  get_schema() -> str
#    Read sqlite_master for table names, then PRAGMA table_info
#    for each table. Return json.dumps(schema).
#
#  run_sql(sql: str) -> str
#    Reject any query that does not start with SELECT.
#    Return results as json.dumps(list_of_dicts).
#    Wrap in try/except — return json.dumps({"error": str(e)}) on failure.
#
#  Test by calling the tools directly (no agent needed):
#    print(get_schema.invoke({}))
#    print(run_sql.invoke({"sql": "SELECT name, dept FROM employees"}))
#    print(run_sql.invoke({"sql": "DELETE FROM employees"}))
#
#  Expected output:
#    {"employees": [{"column": "id", "type": "INTEGER"}, ...], "projects": [...]}
#    [{"name": "Alice", "dept": "Engineering"}, {"name": "Bob", ...}, ...]
#    {"error": "Only SELECT queries are permitted."}
#
# --- starting data ---

In [ ]:
from langchain.tools import tool

# File-based DB so both tool functions can share the same connection target
task2_db: str = os.path.join(tempfile.gettempdir(), "task2_demo.db")

# Seed task2_db with the employee + project data from TASK 1
with sqlite3.connect(task2_db) as _conn:
    _cur = _conn.cursor()
    _cur.execute("DROP TABLE IF EXISTS employees")
    _cur.execute("DROP TABLE IF EXISTS projects")
    _cur.execute("CREATE TABLE employees (id INTEGER PRIMARY KEY, name TEXT, dept TEXT, salary REAL)")
    _cur.execute("CREATE TABLE projects (id INTEGER PRIMARY KEY, name TEXT, budget REAL, lead_id INTEGER)")
    _cur.executemany("INSERT INTO employees VALUES (?,?,?,?)", employee_rows)
    _cur.executemany("INSERT INTO projects VALUES (?,?,?,?)", project_rows)

# ══════════════════════════════════════════════════════════════
#  TASK 3 — WIRE GROQ INTO THE AGENTEXECUTOR
# ══════════════════════════════════════════════════════════════
#
#  Using the tools from TASK 2 and the CONCEPT 4 pattern,
#  build an AgentExecutor backed by Groq and ask it 2 questions.
#
#  Steps:
#    1. Load the Groq API key from the environment
#    2. Build ChatGroq(model="llama-3.1-70b-versatile", temperature=0)
#    3. Build the prompt, agent, and executor (CONCEPT 4 pattern)
#    4. Ask both questions and print result["output"] for each
#    5. If GROQ_API_KEY is not set, print a clear message instead
#
#  Expected output (wording will vary):
#    Q: What is the average salary per department?
#    A: Engineering: $86,500. Finance: $85,000. Sales: $72,000.
#
#    Q: Which project has the highest budget?
#    A: Data Platform has the highest budget at $120,000.
#
# --- starting data ---

In [ ]:
from dotenv import load_dotenv

# Load from the nl_sql_app/.env file if it exists
load_dotenv(os.path.join(os.getcwd(), "nl_sql_app", ".env"))

questions = [
    "What is the average salary per department?",
    "Which project has the highest budget?",
]

# ══════════════════════════════════════════════════════════════
#  TASK 4 — PYDANTIC MODELS FOR THE API CONTRACT
# ══════════════════════════════════════════════════════════════
#
#  FastAPI uses Pydantic models to validate request and response data.
#  Write standalone versions of the models used in backend.py:
#
#  1. SchemaRequest with field: table_name: str
#
#  2. SchemaResponse with fields:
#       table_name: str
#       columns:    list
#       row_count:  int
#
#  3. handle_schema_request(req: SchemaRequest) -> SchemaResponse
#     Query task2_db from TASK 2. Return column names and row count
#     for the requested table.
#
#  Call it twice:
#    handle_schema_request(SchemaRequest(table_name="employees"))
#    handle_schema_request(SchemaRequest(table_name="projects"))
#
#  Expected output:
#    table_name='employees'  columns=['id', 'name', 'dept', 'salary']  row_count=4
#    table_name='projects'   columns=['id', 'name', 'budget', 'lead_id']  row_count=4
#
# --- starting data ---

In [ ]:
from pydantic import BaseModel

# ══════════════════════════════════════════════════════════════
#  CONCEPT 5 -- STREAMLIT: FRONTEND IN PURE PYTHON
# ══════════════════════════════════════════════════════════════
#
#  Streamlit converts Python scripts into interactive web apps.
#  No HTML, no JavaScript, no templates -- just Python functions.
#
#  Execution model:
#    Streamlit re-runs the entire script from top to bottom on
#    every user interaction (button click, text input, selectbox).
#    Use st.session_state to persist values across re-runs.
#
#  Core components used in this project:
#
#    st.title('App title')              -- large heading
#    st.text_input('Label')             -- returns the typed string
#    st.button('Ask')                   -- returns True when clicked
#    st.dataframe(df)                   -- renders a sortable table
#    st.code(sql, language='sql')       -- syntax-highlighted code block
#    st.sidebar.text('sidebar text')    -- content in the left panel
#    st.error('message')                -- red error box
#    st.spinner('Loading...')           -- loading indicator
#
#  pip install streamlit
#  Run: streamlit run app.py    (opens http://localhost:8501)
#
#  EXAMPLE ----------------------------------------------------------

In [ ]:
# ── Streamlit app structure -- simulated in-notebook ───────────
# In real use, this code lives in nl_sql_app/app.py and is
# launched with: streamlit run nl_sql_app/app.py
#
# The simulation below shows how data flows through the app
# without needing an actual Streamlit process.

import json

def simulate_ask_question(question: str) -> dict:
    """Simulates what app.py calls via requests.post('/query')."""
    if not question.strip():
        return {'error': 'Question cannot be empty.'}
    return {
        'sql':     "SELECT * FROM employees WHERE dept = 'Engineering'",
        'results': [{'name': 'Alice', 'dept': 'Engineering', 'salary': 95000},
                    {'name': 'Dave',  'dept': 'Engineering', 'salary': 78000}],
        'latency_ms': 842,
    }


def simulate_streamlit_page(question: str) -> None:
    """
    Represents the Streamlit app flow.
    In app.py, each print() call is replaced by the matching st. call.
    """
    print('[st.title]      NL-SQL: Talk to Any Database')
    print('[st.sidebar]    Backend: http://localhost:8000')
    print()
    print(f"[st.text_input] Question entered: '{question}'")

    if not question.strip():
        print('[st.warning]    Please enter a question.')
        return

    print('[st.spinner]    Asking the AI agent...')
    response = simulate_ask_question(question)

    if 'error' in response:
        print(f'[st.error]      {response["error"]}')
        return

    print(f'[st.success]    Answer ready in {response["latency_ms"]} ms')
    print(f'[st.code/sql]   {response["sql"]}')
    print(f'[st.dataframe]  {json.dumps(response["results"], indent=2)}')
    print(f'[st.caption]    Latency: {response["latency_ms"]} ms')


# ── Run the simulation ────────────────────────────────────────
simulate_streamlit_page('Who are the highest paid engineers?')
print()
simulate_streamlit_page('')   # empty input -- shows the warning path

# ══════════════════════════════════════════════════════════════
#  TASK 5 -- BUILD THE STREAMLIT FRONTEND
# ══════════════════════════════════════════════════════════════
#
#  Create nl_sql_app/app.py -- a Streamlit app that queries your
#  FastAPI backend and displays the results.
#
#  The app must:
#
#  1. Title and sidebar
#       st.title('NL-SQL: Talk to Any Database')
#       st.sidebar.header('Settings')
#       st.sidebar.text(f'Backend: {BACKEND_URL}')
#
#  2. Question input and button
#       question = st.text_input('Ask a question about the data:')
#       clicked  = st.button('Ask')
#
#  3. On button click:
#       - POST to {BACKEND_URL}/query with {'question': question}
#       - If HTTP error: st.error(response.text) and stop
#
#  4. Display results
#       data = response.json()
#       st.code(data['sql'], language='sql')
#       st.dataframe(pd.DataFrame(data['results']))
#       st.caption(f"Latency: {data['latency_ms']} ms")
#
#  5. Error handling
#       Wrap the request in try/except requests.exceptions.ConnectionError
#       If backend is not running: st.error('Backend not reachable...')
#
#  BACKEND_URL is read from the environment:
#    BACKEND_URL = os.environ.get('BACKEND_URL', 'http://localhost:8000')
#
#  Test:
#    uvicorn nl_sql_app.backend:app --reload --port 8000   (Terminal 1)
#    streamlit run nl_sql_app/app.py                        (Terminal 2)
#    Open http://localhost:8501 and ask: 'Who has the highest salary?'
#
# --- starting data ---

In [ ]:
# ask_question() is the function your Streamlit app calls.
# Write it inside nl_sql_app/app.py -- not in this notebook.
import os
import requests

BACKEND_URL = os.environ.get('BACKEND_URL', 'http://localhost:8000')

def ask_question(question: str) -> dict:
    """POST a question to the FastAPI backend and return the response dict."""
    resp = requests.post(f'{BACKEND_URL}/query', json={'question': question}, timeout=30)
    resp.raise_for_status()
    return resp.json()

# ══════════════════════════════════════════════════════════════
#  HOW TO RUN THE FULL APPLICATION
# ══════════════════════════════════════════════════════════════
#
#  Terminal 1 — Seed the database (run once):
#    cd Curriculum/Week_12_Capstone
#    python nl_sql_app/seed_data.py
#
#  Terminal 2 — Start the backend:
#    cd Curriculum/Week_12_Capstone
#    uvicorn nl_sql_app.backend:app --reload --port 8000
#
#    Visit http://localhost:8000/docs for auto-generated API docs.
#    Test /health and /schema directly in the browser.
#
#  Terminal 3 — Start the frontend:
#    cd Curriculum/Week_12_Capstone/nl_sql_app
#    streamlit run app.py
#
#    Streamlit opens at http://localhost:8501
#
#  Sample questions to try:
#    - How many users are there per country?
#    - What is the total revenue per product?
#    - Which user has spent the most money?
#    - List all orders above $500 with the customer name.
#    - What is the average order amount per city?
#    - How many orders were placed in January 2024?
#
#  W12 Royal Road Standard — Observability:
#    Every /query call logs:
#      question received, SQL generated, row count, latency in ms.
#    Latency is also returned in the API response and shown in
#    the Streamlit UI footer.
#
# ══════════════════════════════════════════════════════════════